# Анализ результатов A/B-тестирования по изменению интерфейса онлайн-магазина

<div style="background-color: #ffcc00;">
❗В этом проекте я не буду останавливаться на проверке и предобработке данных. Пример этого есть в другом проекте, который называется Preprocessing data for analysis. А здесь подробно описано, как я оценивала корректность проведённого A/B-тестирования и оценивала его результаты.
</div>

К нам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: как эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Моя задача — провести оценку результатов A/B-теста. В моём распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Нужно оценить корректность проведения теста и проанализировать его результаты.

## Цель исследования

Цель исследования - проверить корректность проведения A/B-тестирования и проанализировать его результаты. 

## Описание данных

Предыдущий аналитик проверял полное обновление дизайна сайта. Гипотеза заключается в следующем: упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации в системе конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта.

Параметры теста:
- название теста: interface_eu_test;
- группы: А (контрольная), B (новый интерфейс).

Нужно:
- загрузить данные теста;
- проверить корректность его проведения;
- проанализировать полученные результаты.
    
Данные собраны в двух файлах - о пользователях и о событиях пользователей.

Структура файла с данными о пользователях:
- `user_id` — идентификатор пользователя;
- `group` — группа пользователя;
- `ab_test` — название теста;
- `device` — устройство, с которого происходила регистрация.

Структура файла с данными о событиях: 
- `user_id` — идентификатор пользователя;
- `event_dt` — дата и время события;
- `event_name` — тип события;
- `details` — дополнительные данные о событии.

---

## Загрузка данных

In [3]:
# Посмотрим, как выглядят данные из первой таблицы
display(participants.head())
participants.info()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [4]:
# Посмотрим, какие A/B-тесты проводятся 
print(participants['ab_test'].unique())

['interface_eu_test' 'recommender_system_test']


In [5]:
# Посмотрим, как выглядят данные из второй таблицы
display(events.head())
events.info()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [6]:
# Посмотрим, какие типы событий есть во второй таблице
print(events['event_name'].unique())

['End of Black Friday Ads Campaign' 'registration' 'product_page' 'login'
 'product_cart' 'purchase' 'Start of Christmas&New Year Promo'
 'Start of CIS New Year Gift Lottery']


---

## Оценка корректности проведения теста

Проверим данные о пользователях по по таблице `participants`:
- одинаковые ли размеры у групп A и B;
- есть ли дубликаты данных в каждой группе;
- есть ли пользователи, которые состоят одновременно в обеих группах или принимают участие одновременно в двух A/B-тестах.

In [7]:
# Выделим из таблицы participants данные, которые относятся к тесту interface_eu_test
participants_interface = participants[participants['ab_test'] == 'interface_eu_test']

# Выделим данные пользователей из двух групп
group_a = participants_interface[participants_interface['group'] == 'A']
group_b = participants_interface[participants_interface['group'] == 'B']

# Проверим количество пользователей в каждой группе
users_count_a = group_a['user_id'].count()
users_count_b = group_b['user_id'].count()

print(f'В группе A - {users_count_a} пользователей, в группе B - {users_count_b} пользователей.', end='\n\n')

users_persentage_a = round(users_count_a / (users_count_a + users_count_b), 2)
users_persentage_b = round(users_count_b / (users_count_a + users_count_b), 2)

print('Соотношение групп:')
print('Группа A: ', users_persentage_a)
print('Группа B: ', users_persentage_b)

В группе A - 5383 пользователей, в группе B - 5467 пользователей.

Соотношение групп:
Группа A:  0.5
Группа B:  0.5


In [8]:
# Проверим все дубли в таблице с нашим тестом
print('Дублей user_id в participants_interface:', participants_interface['user_id'].duplicated().sum())

# Проверим явные и неявные дубликаты в группе A
duplicates_a = group_a.duplicated().sum()
implicit_duplicates_a = group_a['user_id'].duplicated().sum()

# Проверим явные и неявные дубликаты в группе B
duplicates_b = group_b.duplicated().sum()
implicit_duplicates_b = group_b['user_id'].duplicated().sum()

print(f'''Явных дубликатов в группе A - {duplicates_a}, неявных (только по идентификаторам) - {implicit_duplicates_a}.
Явных дубликатов в группе B - {duplicates_b}, неявных - {implicit_duplicates_b}.''')

Дублей user_id в participants_interface: 0
Явных дубликатов в группе A - 0, неявных (только по идентификаторам) - 0.
Явных дубликатов в группе B - 0, неявных - 0.


In [9]:
# Проверим, есть ли пользователи, которые состоят в обеих группах одновременно
intersection = list(set(group_a['user_id']) & set(group_b['user_id']))

print('Количество пересекающихся пользователей в двух группах:', len(intersection))

Количество пересекающихся пользователей в двух группах: 0


In [10]:
# Проверим пересечения тестовой группы нашего теста и тестовой группы конкурирующего теста
participants_recommender = participants[participants['ab_test'] == 'recommender_system_test']

group_b_recommender = participants_recommender[participants_recommender['group'] == 'B']

intersection_b = list(set(group_b['user_id']) & set(group_b_recommender['user_id']))

print('Количество пересекающихся пользователей в тестовых группах конкурирующих тестов: ', len(intersection_b))

Количество пересекающихся пользователей в тестовых группах конкурирующих тестов:  116


In [11]:
# Оставим только тех пользователей, которые не принимают участие во втором тесте
participants_interface = participants_interface[~participants_interface['user_id'].isin(intersection_b)]

**Вывод:** данные соответствуют требованиям технического задания, размеры групп распределены равномерно, между ними пересечений нет. Однако есть 116 пользователей, которые включены в тестовые группы одновременно двух конкурирующих тестов.

---

Далее проанализируем данные о пользовательской активности по таблице `events`:
- оставим только события, которые связаны с пользователями, которые участвуют в нашем тесте;
- проверим дубликаты событий и удалим их при необходимости.

In [12]:
# Оставим только события, которые связаны с пользователями, которые участвуют в нашем тесте
events = events[events['user_id'].isin(participants_interface['user_id'])]

In [13]:
# Проверим дубликаты по событиям и удалим их, если они есть
print('Количество дубликатов по событиям: ', events.duplicated().sum())
events = events.drop_duplicates()

Количество дубликатов по событиям:  6131


Определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации.

In [14]:
# Создадим объект с датами регистрации для каждого пользователя
regs = (events[events['event_name'] == 'registration']
        .groupby('user_id')['event_dt']
        .min()
        .rename('reg_dt'))

# Привяжем даты регистрации к каждому пользователю в исходной таблице
events = events.merge(regs, on='user_id', how='left')

# Рассчитаем количество дней с момента регистрации до совершения события
events['days_after_reg'] = (events['event_dt'] - events['reg_dt']).dt.days

# Оставим только события первых 7 дней
events_7d = events[events['days_after_reg'] < 7]

---

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. 

Заданные параметры:
- базовый показатель конверсии — 30%,
- мощность теста — 80%,
- достоверность теста — 95%.

In [15]:
# Посчитаем, о каком количестве пользователей у нас в итоге есть записи
print('Количество пользователей для исследования: ', len(events_7d['user_id'].unique()))

Количество пользователей для исследования:  10734


С помощью калькулятора [Sample Size Calculator](https://www.evanmiller.org/ab-testing/sample-size.html) рассчитаем, что для проведения A/B-тестирования при таких условиях достаточно выборки в 3692 пользователей для каждой группы - то есть 7384 пользователя всего. У нас же есть информация о 10734 пользователях - этого достаточно для проверки результатов теста.

---

Рассчитаем для каждой группы:
- количество посетителей, сделавших покупку;
- общее количество посетителей.

In [16]:
# Присоединим к получившейся таблице информацию о группах пользователей из исходной таблицы participants
events_7d = events_7d.merge(participants[['user_id', 'group']], on='user_id', how='left')
display(events_7d.head())

,user_id,event_dt,event_name,details,reg_dt,days_after_reg,group
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0,A
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0,A
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0,B
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0,A
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0,B


In [17]:
# Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей
total = events_7d.groupby('group')['user_id'].nunique()
buyers = events_7d[events_7d['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()

conversion = pd.DataFrame({'total' : total, 'buyers': buyers})
conversion['cr'] = buyers / total

print(conversion)

       total  buyers        cr
group                         
A       5723    1579  0.275904
B       5457    1605  0.294118


**Предварительный вывод:** Конверсия тестовой группы выросла на 2 процентных пункта по сравнению с контрольной.

---

## Оценка результатов A/B-тестирования и вывод

Проверим изменение конверсии подходящим статистическим тестом.

In [18]:
# Проведём z-тест, так как он лучше всего подходит для проверки долевых метрик

m_a = conversion.loc['A', 'buyers']   # купили в A
m_b = conversion.loc['B', 'buyers']   # купили в B
n_a = conversion.loc['A', 'total']    # всего в A
n_b = conversion.loc['B', 'total']    # всего в B 

alpha = 0.05

stat_ztest, p_value_ztest = proportions_ztest(
    [m_a, m_b],
    [n_a, n_b],
    alternative='smaller' # так как H_1: p_a < p_b
)

if p_value_ztest > alpha:
    print(f'p-value ({p_value_ztest}) > alpha ({alpha})')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'p-value ({p_value_ztest}) < alpha ({alpha})')
    print('Нулевая гипотеза не находит подтверждения!')

p-value (0.01646497719910885) < alpha (0.05)
Нулевая гипотеза не находит подтверждения!


**Выводы:** Судя по статистике, нулевая гипотеза не подтверждается, а значит мы можем рассматривать результат как заслуживающий доверия. Конверсия в тестовой группе действительно выросла, однако не на 3 процентных пункта, каким был ожидаемый эффект, а на 2. 

Возможная рекомендация для улучшения метрики: сделать воронку событий (registration → login → product_page → purchase), посмотреть, на каком шаге теряется больше всего пользователей, и подумать над упрощением или улучшением интерфейса сайта на этом шаге.